# Sensory + EXN -> EXN + PN combined connectivity matrix

Builds a single presynaptic x postsynaptic connectivity matrix for random-walk
analysis, combining:

- `matrix/exn_pn_matrix.csv` — 177 excitatory neurons (EXN, presynaptic) x 177 EXN + 46
  projection neurons (PN) (postsynaptic).
- `matrix/sensory_to_exn_pn_matrix_0805.csv` — 161 sensory neurons (presynaptic) x the
  same 223 EXN + PN (postsynaptic).

Both matrices' root IDs are remapped to their latest current segmentation ID via
`chunkedgraph.get_latest_roots`, then tagged with cell type from
`spinalcord_neuron_clusters_2` (EXN/PN) and `sensory_neuron_label` (sensory). Sensory
neurons tagged `ab-ltmr` are dropped (161 -> 136). The result is a single
(136 sensory + 177 EXN) x (177 EXN + 46 PN) matrix, saved alongside a root-ID/cell-type
label table for every neuron involved.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import caveclient

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")

BASE_DIR = Path("/Users/wangchu/connectivity_analysis/Random_walk_layer_analysis")
MATRIX_DIR = BASE_DIR / "matrix"

EXN_TAGS = {"adltmr_module", "cltmr_module", "noci_module", "np_module", "r", "v1", "v2"}
PN_TAGS = {"p1", "p2_cold", "p2_noci", "p2_poly"}
DROP_SENSORY_TAGS = {"ab-ltmr"}


## 1) Load the raw matrices

IDs are kept as strings throughout to avoid int/float/str join mismatches. Both files
are presynaptic (rows) x postsynaptic (columns); the 223 columns are the same neuron
set in both files, just in different column order.


In [2]:
exn_pn_df = pd.read_csv(MATRIX_DIR / "exn_pn_matrix.csv", index_col=0)
sensory_df = pd.read_csv(MATRIX_DIR / "sensory_to_exn_pn_matrix_0805.csv", index_col=0)

exn_pn_df.index = exn_pn_df.index.astype(str)
exn_pn_df.columns = exn_pn_df.columns.astype(str)
sensory_df.index = sensory_df.index.astype(str)
sensory_df.columns = sensory_df.columns.astype(str)

print("exn_pn_matrix.csv:                 ", exn_pn_df.shape)
print("sensory_to_exn_pn_matrix_0805.csv: ", sensory_df.shape)
assert set(exn_pn_df.columns) == set(sensory_df.columns), "the two files' 223 postsynaptic columns should be the same neuron set"


exn_pn_matrix.csv:                  (177, 223)
sensory_to_exn_pn_matrix_0805.csv:  (161, 223)


## 2) Current cell-type tags from CAVE

`spinalcord_neuron_clusters_2` tags EXN + PN (223 rows); `sensory_neuron_label` tags
sensory neurons. A root ID can appear more than once (re-tagging); keep the most
frequent tag per ID. Tags are stripped of whitespace (the raw table has entries like
`"mrgd "`, `"sst "`).


In [3]:
def load_tag_table(table_name):
    df = client.materialize.query_table(table_name)
    df = df[["pt_root_id", "tag"]].dropna()
    df["pt_root_id"] = df["pt_root_id"].astype(str).str.strip()
    df["tag"] = df["tag"].astype(str).str.strip()
    id_to_tag = df.groupby("pt_root_id")["tag"].agg(lambda s: s.value_counts().index[0])
    return id_to_tag

exn_pn_id_to_tag = load_tag_table("spinalcord_neuron_clusters_2")
sensory_id_to_tag = load_tag_table("sensory_neuron_label")

print(f"{len(exn_pn_id_to_tag)} tagged EXN/PN neurons in spinalcord_neuron_clusters_2")
print(exn_pn_id_to_tag.value_counts())
print()
print(f"{len(sensory_id_to_tag)} tagged sensory neurons in sensory_neuron_label")
print(sensory_id_to_tag.value_counts())


Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.
Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


223 tagged EXN/PN neurons in spinalcord_neuron_clusters_2
tag
v1               41
v2               36
np_module        27
cltmr_module     24
r                24
p2_noci          16
adltmr_module    15
p2_poly          15
noci_module      10
p2_cold           9
p1                6
Name: count, dtype: int64

161 tagged sensory neurons in sensory_neuron_label
tag
C-COLD      25
ab-ltmr     25
ad-htmr     21
c-htmr-p    20
adltmr      18
mrgd        18
sst         18
cltmr       16
Name: count, dtype: int64


## 3) Remap stale root IDs to their latest root, then to a tag

IDs in the two CSVs may predate later proofreading edits, so they might not appear
directly in the tag table. For any ID not found directly, ask
`chunkedgraph.get_latest_roots` for its current segment ID(s) and use whichever of
those *is* in the tag table. IDs that still can't be resolved (no current root in the
tag table, or more than one match) are reported and dropped.


In [4]:
def resolve_ids(ids, id_to_tag, desc):
    """Return (resolved_tag, id_remap, unresolved) for `ids` against `id_to_tag`.

    resolved_tag: old_id -> tag (covers both directly-tagged and remapped ids)
    id_remap: stale_id -> current_id actually used for the tag lookup
    unresolved: list of (old_id, reason)
    """
    ids = sorted(set(ids))
    tagged_ids = set(id_to_tag.index)
    resolved_tag = {i: id_to_tag[i] for i in ids if i in tagged_ids}
    id_remap = {}
    unresolved = []

    outdated_ids = [i for i in ids if i not in tagged_ids]
    print(f"[{desc}] {len(ids)} unique IDs, {len(outdated_ids)} not directly tagged; resolving via chunkedgraph")

    for old_id in tqdm(outdated_ids, desc=desc):
        try:
            latest_roots = client.chunkedgraph.get_latest_roots(int(old_id))
        except Exception as e:
            unresolved.append((old_id, f"chunkedgraph error: {e}"))
            continue

        latest_roots = [str(int(r)) for r in latest_roots]
        latest_in_tag_table = [r for r in latest_roots if r in tagged_ids]

        if len(latest_in_tag_table) == 1:
            new_id = latest_in_tag_table[0]
            id_remap[old_id] = new_id
            resolved_tag[old_id] = id_to_tag[new_id]
        elif len(latest_in_tag_table) == 0:
            unresolved.append((old_id, f"no latest root in tag table (latest roots: {latest_roots})"))
        else:
            unresolved.append((old_id, f"ambiguous: multiple latest roots in tag table {latest_in_tag_table}"))

    print(f"  resolved via chunkedgraph: {len(id_remap)}; unresolved: {len(unresolved)}")
    for old_id, reason in unresolved[:20]:
        print(f"    {old_id}: {reason}")
    return resolved_tag, id_remap, unresolved


In [5]:
exn_pn_universe_ids = set(exn_pn_df.index) | set(exn_pn_df.columns) | set(sensory_df.columns)
exn_pn_resolved_tag, exn_pn_id_remap, exn_pn_unresolved = resolve_ids(
    exn_pn_universe_ids, exn_pn_id_to_tag, "EXN/PN"
)


EXN/PN:   0%|          | 0/2 [00:00<?, ?it/s]

[EXN/PN] 223 unique IDs, 2 not directly tagged; resolving via chunkedgraph


EXN/PN: 100%|██████████| 2/2 [00:01<00:00,  1.41it/s]

  resolved via chunkedgraph: 2; unresolved: 0


In [6]:
sensory_universe_ids = set(sensory_df.index)
sensory_resolved_tag, sensory_id_remap, sensory_unresolved = resolve_ids(
    sensory_universe_ids, sensory_id_to_tag, "sensory"
)


sensory:   0%|          | 0/2 [00:00<?, ?it/s]

[sensory] 161 unique IDs, 2 not directly tagged; resolving via chunkedgraph


sensory: 100%|██████████| 2/2 [00:00<00:00,  4.10it/s]

  resolved via chunkedgraph: 2; unresolved: 0


## 4) Apply the remap and drop unresolved neurons

`current_id_of[old_id]` gives the current root ID to use everywhere downstream (itself,
if it was already current). IDs with no resolved tag are dropped from the matrices
(reported above).


In [7]:
exn_pn_current_id_of = {i: exn_pn_id_remap.get(i, i) for i in exn_pn_resolved_tag}
sensory_current_id_of = {i: sensory_id_remap.get(i, i) for i in sensory_resolved_tag}

def remap_axis(df, axis_current_id_of, axis):
    ids = df.index if axis == 0 else df.columns
    keep = [i for i in ids if i in axis_current_id_of]
    df = df.loc[keep] if axis == 0 else df[keep]
    new_ids = [axis_current_id_of[i] for i in keep]
    if axis == 0:
        df = df.copy()
        df.index = new_ids
        df = df.groupby(level=0).sum()  # merge any old ids that now share a current id
    else:
        df = df.copy()
        df.columns = new_ids
        df = df.T.groupby(level=0).sum().T
    return df

exn_pn_df = remap_axis(exn_pn_df, exn_pn_current_id_of, axis=0)   # rows: EXN (presynaptic)
exn_pn_df = remap_axis(exn_pn_df, exn_pn_current_id_of, axis=1)   # cols: EXN + PN (postsynaptic)

sensory_df = remap_axis(sensory_df, sensory_current_id_of, axis=0)  # rows: sensory (presynaptic)
sensory_df = remap_axis(sensory_df, exn_pn_current_id_of, axis=1)   # cols: EXN + PN (postsynaptic)

print("exn_pn_matrix after remap:  ", exn_pn_df.shape)
print("sensory_matrix after remap:", sensory_df.shape)
assert set(exn_pn_df.columns) == set(sensory_df.columns), "postsynaptic column sets diverged after remap"
assert set(exn_pn_df.index) <= set(exn_pn_df.columns), "EXN presynaptic ids should all be present among the postsynaptic columns"


exn_pn_matrix after remap:   (177, 223)
sensory_matrix after remap: (161, 223)


## 5) Filter out `ab-ltmr` sensory neurons

Drop sensory neurons tagged `ab-ltmr` from the row set (161 -> expected 136).


In [8]:
sensory_tag_of = pd.Series(sensory_resolved_tag, name="tag")
sensory_tag_of.index = [sensory_current_id_of[i] for i in sensory_tag_of.index]
sensory_tag_of = sensory_tag_of.groupby(level=0).first()
sensory_tag_of = sensory_tag_of.reindex(sensory_df.index)

sensory_keep_ids = sensory_tag_of[~sensory_tag_of.isin(DROP_SENSORY_TAGS)].index.tolist()
n_dropped = sensory_df.shape[0] - len(sensory_keep_ids)
print(f"sensory neurons: {sensory_df.shape[0]} total, {n_dropped} dropped ({sorted(DROP_SENSORY_TAGS)}), {len(sensory_keep_ids)} remaining")

if len(sensory_keep_ids) != 136:
    print(f"WARNING: expected 136 non-ab-ltmr sensory neurons, got {len(sensory_keep_ids)} -- check DROP_SENSORY_TAGS / tag table for changes")


sensory neurons: 161 total, 25 dropped (['ab-ltmr']), 136 remaining


## 6) Assemble the combined matrix

Rows: 136 filtered sensory neurons + 177 EXN (presynaptic). Columns: 177 EXN + 46 PN
(postsynaptic, same 223 neurons as the EXN row set plus PN). Both axes are sorted by
cell-type tag then root ID for readability.


In [9]:
exn_pn_tag_of = pd.Series(exn_pn_resolved_tag, name="tag")
exn_pn_tag_of.index = [exn_pn_current_id_of[i] for i in exn_pn_tag_of.index]
exn_pn_tag_of = exn_pn_tag_of.groupby(level=0).first()

def group_of(tag):
    if tag in EXN_TAGS:
        return "exn"
    if tag in PN_TAGS:
        return "pn"
    return "unknown"

col_tag_of = exn_pn_tag_of.reindex(exn_pn_df.columns)
col_group_of = col_tag_of.map(group_of)
assert (col_group_of == "unknown").sum() == 0, "found postsynaptic tag(s) outside the documented EXN/PN tag sets"

exn_row_ids = exn_pn_df.index.tolist()

def sort_by_tag(ids, tag_of):
    return sorted(ids, key=lambda i: (tag_of.get(i, ""), i))

final_col_order = sort_by_tag(exn_pn_df.columns.tolist(), col_tag_of)
final_sensory_row_order = sort_by_tag(sensory_keep_ids, sensory_tag_of)
final_exn_row_order = sort_by_tag(exn_row_ids, exn_pn_tag_of)
final_row_order = final_sensory_row_order + final_exn_row_order

top_block = sensory_df.loc[final_sensory_row_order, final_col_order]
bottom_block = exn_pn_df.loc[final_exn_row_order, final_col_order]
combined_df = pd.concat([top_block, bottom_block], axis=0)

print("combined matrix:", combined_df.shape, "(expected (136 + 177, 223) = (313, 223))")
combined_df.head()


combined matrix: (313, 223) (expected (136 + 177, 223) = (313, 223))


,720575940878312571,720575940880839442,720575940883729656,720575940885005156,720575940886853257,720575940888165967,720575940890900801,720575940895289006,720575940896732090,720575940907584340,...,720575940913573871,720575940915253823,720575940915256127,720575940916299357,720575940920054001,720575940921525267,720575940922957341,720575940923272236,720575940926242958,720575940939715909
720575940851372659,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940851375219,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940872856946,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940878947380,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940881127565,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7) Cell-type label table

One row per unique neuron across the whole matrix (rows and columns): root ID, cell
type tag, broad group (`sensory` / `exn` / `pn`), and which side(s) of the matrix it
appears on.


In [10]:
label_rows = []

for i in final_sensory_row_order:
    label_rows.append({"root_id": i, "tag": sensory_tag_of[i], "group": "sensory", "matrix_role": "presynaptic"})

for i in final_col_order:
    role = "presynaptic_and_postsynaptic" if i in exn_row_ids else "postsynaptic"
    label_rows.append({"root_id": i, "tag": col_tag_of[i], "group": col_group_of[i], "matrix_role": role})

labels_df = pd.DataFrame(label_rows).drop_duplicates(subset="root_id").reset_index(drop=True)
print(labels_df.shape, "unique neurons (expected 136 sensory + 177 exn + 46 pn = 359)")
labels_df["group"].value_counts()


(359, 4) unique neurons (expected 136 sensory + 177 exn + 46 pn = 359)


group
exn        177
sensory    136
pn          46
Name: count, dtype: int64

## 8) Save outputs


In [11]:
combined_out = BASE_DIR / "sensory_exn_to_exn_pn_matrix.csv"
labels_out = BASE_DIR / "sensory_exn_to_exn_pn_cell_tags.csv"

combined_df.to_csv(combined_out)
labels_df.to_csv(labels_out, index=False)

print("Saved:", combined_out, combined_df.shape)
print("Saved:", labels_out, labels_df.shape)


Saved: /Users/wangchu/connectivity_analysis/Random_walk_layer_analysis/sensory_exn_to_exn_pn_matrix.csv (313, 223)
Saved: /Users/wangchu/connectivity_analysis/Random_walk_layer_analysis/sensory_exn_to_exn_pn_cell_tags.csv (359, 4)
